# 📊 Winosa — Exploratory Data Analysis

Notebook ini menganalisis data dari Winosa backend:
- **Blog**: distribusi views, readTime, tags
- **Subscriber**: growth bulanan, status aktif/nonaktif
- **Contact**: read/unread, panjang pesan

## 0. Setup & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import requests
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# Style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')

os.makedirs('../data', exist_ok=True)

# Toggle: True = pakai sample data lokal, False = fetch dari API
USE_SAMPLE = True
API_BASE   = os.getenv('API_BASE', 'http://localhost:5000/api')
API_TOKEN  = os.getenv('API_TOKEN', '')

print('Setup selesai ✓')
print(f'Mode: {"Sample" if USE_SAMPLE else "Live API"}')

## 1. Load Data

In [ ]:
if USE_SAMPLE:
    # ── Sample Data ──────────────────────────────────────────────
    raw_blogs = [
        {'_id':'b1','title':'Membangun UI Profesional','slug':'ui-pro','tags':['UI/UX','Design'],
         'isPublished':True,'views':320,'readTime':5,'author':'Admin','createdAt':'2024-09-10'},
        {'_id':'b2','title':'Panduan Next.js 14','slug':'nextjs-14','tags':['Technology'],
         'isPublished':True,'views':870,'readTime':9,'author':'Admin','createdAt':'2024-10-02'},
        {'_id':'b3','title':'Strategi Branding 2025','slug':'branding-2025','tags':['Insight','Branding'],
         'isPublished':True,'views':450,'readTime':6,'author':'Admin','createdAt':'2024-11-15'},
        {'_id':'b4','title':'React vs Vue 2025','slug':'react-vue','tags':['Technology'],
         'isPublished':False,'views':0,'readTime':7,'author':'Admin','createdAt':'2024-12-01'},
        {'_id':'b5','title':'Tips SEO untuk Startup','slug':'seo-startup','tags':['Insight','SEO'],
         'isPublished':True,'views':215,'readTime':4,'author':'Admin','createdAt':'2025-01-08'},
        {'_id':'b6','title':'TypeScript Best Practices','slug':'ts-best','tags':['Technology'],
         'isPublished':True,'views':540,'readTime':8,'author':'Admin','createdAt':'2025-02-20'},
        {'_id':'b7','title':'Desain System Guideline','slug':'design-system','tags':['UI/UX','Design'],
         'isPublished':True,'views':180,'readTime':5,'author':'Admin','createdAt':'2025-03-05'},
        {'_id':'b8','title':'Draft: Mobile UX Patterns','slug':'mobile-ux','tags':['UI/UX'],
         'isPublished':False,'views':0,'readTime':6,'author':'Admin','createdAt':'2025-04-01'},
    ]

    raw_subscribers = [
        {'_id':f's{i}','email':f'user{i}@email.com',
         'isActive': i % 5 != 0,
         'createdAt': f'2024-{(i%12)+1:02d}-{(i%28)+1:02d}'}
        for i in range(1, 31)
    ]

    raw_contacts = [
        {'_id':'c1','name':'Budi','email':'budi@email.com','subject':'Harga Jasa','message':'Berapa harga untuk web app?','isRead':True,'createdAt':'2025-01-10'},
        {'_id':'c2','name':'Sari','email':'sari@email.com','subject':None,'message':'Halo, saya tertarik dengan layanan Winosa.','isRead':False,'createdAt':'2025-02-05'},
        {'_id':'c3','name':'Doni','email':'doni@email.com','subject':'Kolaborasi','message':'Apakah Winosa menerima kemitraan untuk proyek jangka panjang?','isRead':True,'createdAt':'2025-02-18'},
        {'_id':'c4','name':'Rina','email':'rina@email.com','subject':'Portfolio','message':'Bisa share portfolio terbaru?','isRead':False,'createdAt':'2025-03-01'},
        {'_id':'c5','name':'Agus','email':'agus@email.com','subject':None,'message':'Butuh revisi desain untuk landing page kami yang sudah ada, apakah bisa?','isRead':False,'createdAt':'2025-03-22'},
    ]
    print('Sample data loaded ✓')

else:
    # ── Live API ──────────────────────────────────────────────────
    headers = {'Authorization': f'Bearer {API_TOKEN}'}
    raw_blogs       = requests.get(f'{API_BASE}/admin/blog',          headers=headers).json().get('data', [])
    raw_subscribers = requests.get(f'{API_BASE}/newsletter',          headers=headers).json().get('data', [])
    raw_contacts    = requests.get(f'{API_BASE}/contact',             headers=headers).json().get('data', [])
    print(f'Live data loaded — blogs:{len(raw_blogs)} subscribers:{len(raw_subscribers)} contacts:{len(raw_contacts)}')

## 2. Parse ke DataFrame

In [ ]:
# ── Blog ──────────────────────────────────────────────────────────
df_blog = pd.DataFrame(raw_blogs)
df_blog['createdAt']    = pd.to_datetime(df_blog['createdAt'])
df_blog['month']        = df_blog['createdAt'].dt.to_period('M')
df_blog['tag_count']    = df_blog['tags'].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_blog['primary_tag']  = df_blog['tags'].apply(lambda x: x[0] if isinstance(x, list) and x else 'Unknown')

# ── Subscriber ────────────────────────────────────────────────────
df_sub = pd.DataFrame(raw_subscribers)
df_sub['createdAt'] = pd.to_datetime(df_sub['createdAt'])
df_sub['month']     = df_sub['createdAt'].dt.to_period('M')

# ── Contact ───────────────────────────────────────────────────────
df_con = pd.DataFrame(raw_contacts)
df_con['createdAt']     = pd.to_datetime(df_con['createdAt'])
df_con['msg_length']    = df_con['message'].str.len()
df_con['has_subject']   = df_con['subject'].notna()

print(f'Blog       : {len(df_blog)} rows × {df_blog.shape[1]} cols')
print(f'Subscriber : {len(df_sub)} rows × {df_sub.shape[1]} cols')
print(f'Contact    : {len(df_con)} rows × {df_con.shape[1]} cols')
df_blog.head(3)

## 3. Missing Values

In [ ]:
def missing_report(df, label):
    total  = df.isnull().sum()
    pct    = (total / len(df) * 100).round(1)
    report = pd.DataFrame({'Missing': total, 'Pct (%)': pct})
    report = report[report['Missing'] > 0].sort_values('Missing', ascending=False)
    print(f'\n── {label} ──')
    print(report if not report.empty else '  Tidak ada missing values ✓')
    return report

missing_report(df_blog, 'Blog')
missing_report(df_sub,  'Subscriber')
missing_report(df_con,  'Contact')

# Heatmap
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (df, title) in zip(axes, [(df_blog,'Blog'),(df_sub,'Subscriber'),(df_con,'Contact')]):
    sns.heatmap(df.isnull(), ax=ax, cbar=False, yticklabels=False,
                cmap=['#f0f0f0','#EF4444'])
    ax.set_title(f'{title} — Missing Values', fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig('../data/missing_values.png', bbox_inches='tight')
plt.show()
print('Saved → data/missing_values.png')

## 4. Distribusi Blog

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Distribusi Blog', fontsize=14, fontweight='bold', y=1.01)

# 4a — Histogram Views
ax = axes[0, 0]
pub = df_blog[df_blog['isPublished'] == True]['views']
ax.hist(pub, bins=8, color='#EAB308', edgecolor='white', linewidth=0.8)
ax.axvline(pub.mean(), color='#111', linestyle='--', linewidth=1.2, label=f'Mean: {pub.mean():.0f}')
ax.set_title('Distribusi Views (Published)', fontweight='bold')
ax.set_xlabel('Views'); ax.set_ylabel('Jumlah Blog')
ax.legend(fontsize=9)

# 4b — Bar ReadTime
ax = axes[0, 1]
rt = df_blog[['title','readTime']].sort_values('readTime', ascending=False).head(8)
bars = ax.barh(rt['title'].str[:25], rt['readTime'], color='#6366F1', edgecolor='white')
ax.set_title('Read Time per Blog (menit)', fontweight='bold')
ax.set_xlabel('Menit')
ax.invert_yaxis()
for bar in bars:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.0f}m', va='center', fontsize=8)

# 4c — Pie Published vs Draft
ax = axes[1, 0]
counts = df_blog['isPublished'].value_counts()
labels = ['Published' if k else 'Draft' for k in counts.index]
ax.pie(counts, labels=labels, colors=['#22C55E','#9CA3AF'],
       autopct='%1.0f%%', startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Published vs Draft', fontweight='bold')

# 4d — Views per Tag
ax = axes[1, 1]
tag_views = df_blog.groupby('primary_tag')['views'].sum().sort_values(ascending=True)
tag_views.plot(kind='barh', ax=ax, color='#F97316', edgecolor='white')
ax.set_title('Total Views per Primary Tag', fontweight='bold')
ax.set_xlabel('Total Views')

plt.tight_layout()
plt.savefig('../data/blog_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → data/blog_distribution.png')

## 5. Korelasi: ReadTime vs Views

In [ ]:
pub = df_blog[df_blog['isPublished'] == True].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Korelasi Blog', fontsize=13, fontweight='bold')

# 5a — Scatter + trendline
ax = axes[0]
ax.scatter(pub['readTime'], pub['views'], color='#EAB308', edgecolors='#111', linewidths=0.6, s=80, zorder=3)
if len(pub) > 1:
    m, b = np.polyfit(pub['readTime'], pub['views'], 1)
    x_line = np.linspace(pub['readTime'].min(), pub['readTime'].max(), 100)
    ax.plot(x_line, m*x_line + b, color='#EF4444', linewidth=1.5, linestyle='--', label='Trend')
    r = pub['readTime'].corr(pub['views'])
    ax.set_title(f'ReadTime vs Views  (r = {r:.2f})', fontweight='bold')
    ax.legend(fontsize=9)
else:
    ax.set_title('ReadTime vs Views', fontweight='bold')
ax.set_xlabel('Read Time (menit)'); ax.set_ylabel('Views')

# 5b — Correlation heatmap
ax = axes[1]
corr_cols = ['views','readTime','tag_count']
corr_matrix = pub[corr_cols].corr()
sns.heatmap(corr_matrix, ax=ax, annot=True, fmt='.2f', cmap='YlOrBr',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/correlation.png', bbox_inches='tight')
plt.show()
print('Saved → data/correlation.png')

## 6. Subscriber Growth

In [ ]:
monthly = df_sub.groupby('month').size().reset_index(name='new_subscribers')
monthly['month_str']   = monthly['month'].astype(str)
monthly['cumulative']  = monthly['new_subscribers'].cumsum()

active_pct   = df_sub['isActive'].mean() * 100
inactive_pct = 100 - active_pct

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Subscriber Growth', fontsize=13, fontweight='bold')

# 6a — Bar new per bulan + line kumulatif
ax1 = axes[0]
ax2 = ax1.twinx()
bars = ax1.bar(monthly['month_str'], monthly['new_subscribers'],
               color='#EAB308', alpha=0.85, edgecolor='white', label='Baru')
ax2.plot(monthly['month_str'], monthly['cumulative'],
         color='#111', marker='o', markersize=4, linewidth=1.8, label='Kumulatif')
ax1.set_title('Subscriber per Bulan', fontweight='bold')
ax1.set_xlabel('Bulan')
ax1.set_ylabel('Subscriber Baru', color='#EAB308')
ax2.set_ylabel('Total Kumulatif', color='#111')
ax1.tick_params(axis='x', rotation=45, labelsize=8)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

# 6b — Pie active vs inactive
ax = axes[1]
ax.pie([active_pct, inactive_pct],
       labels=[f'Active ({active_pct:.0f}%)', f'Inactive ({inactive_pct:.0f}%)'],
       colors=['#22C55E','#E5E7EB'],
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5),
       autopct='%1.0f%%')
ax.set_title('Status Subscriber', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/subscriber_growth.png', bbox_inches='tight')
plt.show()
print('Saved → data/subscriber_growth.png')

## 7. Analisis Kontak

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Analisis Kontak', fontsize=13, fontweight='bold')

# 7a — Read vs Unread
ax = axes[0]
read_counts = df_con['isRead'].value_counts()
ax.pie(read_counts,
       labels=['Read' if k else 'Unread' for k in read_counts.index],
       colors=['#22C55E','#F97316'],
       autopct='%1.0f%%', startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Read vs Unread', fontweight='bold')

# 7b — Subject vs Tidak
ax = axes[1]
has_sub = df_con['has_subject'].value_counts()
bars = ax.bar(['Punya Subject','Tanpa Subject'],
              [has_sub.get(True,0), has_sub.get(False,0)],
              color=['#6366F1','#9CA3AF'], edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Kelengkapan Subject', fontweight='bold')
ax.set_ylabel('Jumlah')

# 7c — Histogram panjang pesan
ax = axes[2]
ax.hist(df_con['msg_length'], bins=6, color='#EAB308', edgecolor='white')
ax.axvline(df_con['msg_length'].mean(), color='#111', linestyle='--', linewidth=1.2,
           label=f'Mean: {df_con["msg_length"].mean():.0f} chars')
ax.set_title('Panjang Pesan', fontweight='bold')
ax.set_xlabel('Jumlah Karakter'); ax.set_ylabel('Frekuensi')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/contact_analysis.png', bbox_inches='tight')
plt.show()
print('Saved → data/contact_analysis.png')

## 8. Summary Statistics

In [ ]:
print('═' * 50)
print('BLOG SUMMARY')
print('═' * 50)
print(df_blog[['views','readTime','tag_count']].describe().round(1))

pub = df_blog[df_blog['isPublished'] == True]
if not pub.empty:
    top_blog = pub.loc[pub['views'].idxmax()]
    top_tag  = pub.groupby('primary_tag')['views'].sum().idxmax()
    r        = pub['readTime'].corr(pub['views'])
    print(f'\n📌 Blog terpopuler : {top_blog["title"]} ({top_blog["views"]} views)')
    print(f'📌 Tag terbaik     : {top_tag}')
    print(f'📌 Korelasi readTime↔views : {r:.2f} ({"positif" if r>0 else "negatif"})')

print('\n' + '═' * 50)
print('SUBSCRIBER SUMMARY')
print('═' * 50)
print(f'Total subscriber : {len(df_sub)}')
print(f'Active           : {df_sub["isActive"].sum()} ({df_sub["isActive"].mean()*100:.0f}%)')
print(f'Inactive         : {(~df_sub["isActive"]).sum()}')
if len(monthly) >= 2:
    trend = monthly['new_subscribers'].iloc[-1] - monthly['new_subscribers'].iloc[-2]
    print(f'Tren bulan terakhir : {trend:+d} vs bulan sebelumnya')

print('\n' + '═' * 50)
print('CONTACT SUMMARY')
print('═' * 50)
print(f'Total kontak     : {len(df_con)}')
print(f'Belum dibaca     : {(~df_con["isRead"]).sum()}')
print(f'Rata panjang msg : {df_con["msg_length"].mean():.0f} karakter')
print(f'Ada subject      : {df_con["has_subject"].sum()} dari {len(df_con)}')

print('\n✅ EDA selesai. Semua chart tersimpan di folder data/')